# 14 - Paired bootstrap for Passes 5-7 (CPU only, no GPU)

Trains nothing, so it needs **no GPU**: on Kaggle set **Accelerator: None**. About 10-20 minutes. Code: `src/gainsplit/analysis_pass5_7.py` (reuses the bootstrap of `analysis.py` and `analysis_pass4.py`).

**Inputs (right-hand panel, Add Input):** only **`gainsplit-processed`** (any version with `isarcasm_test.csv`; it is used for the word-count baseline). All run predictions come from the repo, so `results/pass5`, `pass6` and `pass7` must be committed and pushed first.

**Internet: on** (clones the repo with the code and every committed run).

Every contrast was fixed before this notebook was run (vault: Experiment Plan, "Bootstrap plan for Passes 5-7"). Paired bootstrap over test items, 10,000 resamples, the same resample for every seed, statistic = mean over seeds 0-4. Holm correction within each table. Scores in points. Tables go to `/kaggle/working/analysis_pass5_7/`; download `analysis_pass5_7.zip` at the end.

| Part | Question |
|---|---|
| A. Checks | Do all re-runs equal their originals? Does every two-stage run's stage 1 equal Pass 4's A15? |
| B. In-domain contrasts | Pass 5 (pair weight, cleaning, FGM), Pass 6 (two-stage) and Pass 7 (all labels): test macro-F1, and accuracy at 100% |
| C. Transfer | iSarcasmEval AUC of each pair arm vs its baseline, and vs **word count alone** on the same items |
| D. Outlet shortcut | Cue-mask gap of the Pass 5-6 arms vs their baselines |

## 1. Get the code and find the inputs

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    REPO_DIR = "/tmp/GainSplit"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Ariffurrhmn/GainSplit.git", REPO_DIR], check=True)
    INPUT = Path("/kaggle/input")
    OUT_DIR = Path("/kaggle/working/analysis_pass5_7")
else:  # local run from the repo
    REPO_DIR = str(Path.cwd().parent) if Path.cwd().name == "experiments" else str(Path.cwd())
    INPUT = Path(REPO_DIR) / "data" / "kaggle_upload"
    OUT_DIR = Path(REPO_DIR) / "results" / "analysis_pass5_7"
sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["GAINSPLIT_INPUT"] = str(INPUT)
import pandas as pd
pd.set_option("display.width", 200)
from gainsplit.data import find_data_dir, verify
from gainsplit import analysis as an, analysis_pass4 as a4, analysis_pass5_7 as a57

print("code commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
DATA_DIR = find_data_dir(str(INPUT))
verify(DATA_DIR)
R = Path(REPO_DIR) / "results"
RESULTS_DIRS = [R / p for p in ("pass1", "pass2a", "pass2b", "pass3", "pass4", "pass5", "pass6", "pass7")]
missing = [str(p) for p in RESULTS_DIRS if not (p / "runs").is_dir()]
assert not missing, f"missing results folders (commit and push them first): {missing}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
N_BOOT, RNG_SEED = 10000, 0
isarc_test = pd.read_csv(DATA_DIR / "isarcasm_test.csv", keep_default_na=False)
print("data:", DATA_DIR, "| iSarcasmEval test items:", len(isarc_test))

## A. Load the runs and check them (read this first)
- **Re-run check:** every re-run (tags `mask`, `explain`, `rerun`) must give the same test scores and probabilities as its original (`ok` = True). Expected: 41 rows, all True.
- **Stage-1 check:** every A18/A19 run's stage-1 dev macro-F1 must equal Pass 4's A15 at 10% for the same seed. Expected: 25 rows, all True.

If either check fails, stop and report: the contrasts against older baselines would not be clean.

In [ ]:
runs = an.load_runs(*RESULTS_DIRS)
print(len(runs), "runs loaded")
rr = a4.check_reruns(runs, tags=("mask", "explain", "rerun"))
s1 = a57.check_stage1(runs)
print("re-runs identical:", len(rr), "rows,", bool(len(rr)) and bool(rr["ok"].all()))
print("stage 1 equals A15:", len(s1), "rows,", bool(len(s1)) and bool(s1["ok"].all()))
display(a57.summary(runs))

## B. In-domain contrasts (test, paired bootstrap)
`mean_diff` in points, 95% interval, two-sided `p`, and `p_holm` (Holm over all rows of the table). Read `p_holm` for claims.
- **Pass 5:** does weight 0.1 recover the loss (vs A10 at weight 1.0), is it still below A1, does cleaning add anything, does FGM add on top?
- **Pass 6:** do the two-stage arms match A1 / A7 (10%) or lose (1%)?
- **Pass 7:** does FGM help at 100% labels, and does the pair stage cost anything there? Accuracy is shown too, since that is the number the seed paper reports.

In [ ]:
f1 = a57.contrast_table(runs, "macro_f1", N_BOOT, RNG_SEED)
display(f1.round(4))
acc = a57.contrast_table(runs, "accuracy", N_BOOT, RNG_SEED)
display(acc.round(4))

## C. Transfer to iSarcasmEval
- **AUC contrasts:** pair arm minus its baseline (points).
- **AUC vs word count:** pair arm minus the AUC of word count alone, on the same items (Pass 4 found word count alone gets about 68.7). This is the transfer claim net of length: a positive interval that excludes 0 after Holm means the arm transfers better than length alone.

In [ ]:
auc = a57.auc_table(runs, N_BOOT, RNG_SEED)
display(auc.round(4))
alen = a57.length_table(runs, isarc_test, N_BOOT, RNG_SEED)
display(alen.round(4))

## D. Outlet shortcut (cue-mask gap)
Gap = random-mask minus cue-mask macro-F1. `mean_diff` < 0 means the pair arm leans less on outlet cue words than its baseline. Pass 7 is left out: A1 at 100% has masked predictions for one seed only.

In [ ]:
gap = a57.gap_table(runs, N_BOOT, RNG_SEED)
display(gap.round(4))

## E. Save every table and zip them

In [ ]:
tables = {"summary": a57.summary(runs), "rerun_check": rr, "stage1_check": s1, "contrasts_f1": f1,
          "contrasts_accuracy_100": acc, "auc_contrasts": auc, "auc_vs_length": alen, "gap_contrasts": gap}
for name, t in tables.items():
    if len(t):
        an.save_table(t, OUT_DIR, name, default_decimals=3)
print("tables:", [k for k, v in tables.items() if len(v)])
if ON_KAGGLE:
    print(shutil.make_archive("/kaggle/working/analysis_pass5_7", "zip", OUT_DIR))